In [1]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# 1. 定义状态 (移除了 image_paths)
class AgentState(TypedDict):
    system: str
    user: str
    response: str

# 2. 初始化模型
llm = ChatOllama(
    model="qwen3.5",  # 哪怕是VL模型，也可以只传纯文字
    base_url="http://110.42.252.68:8080",
    temperature=0.9
)

# 3. 定义处理节点 (简化为纯文本对话)
def chat_node(state: AgentState):
    messages = [
        SystemMessage(content=state["system"]),
        HumanMessage(content=state["user"])
    ]
    
    # 直接调用，无需组装复杂的多模态 Content 结构
    res = llm.invoke(messages)
    
    return {"response": res.content}

# 4. 构建图
workflow = StateGraph(AgentState)
workflow.add_node("chat_agent", chat_node)
workflow.add_edge(START, "chat_agent")
workflow.add_edge("chat_agent", END)
app = workflow.compile()

# --- 调用示例 ---
if __name__ == "__main__":
    inputs = {
        "system": "你是一个专业的AI助手。",
        "user": "请解释一下LangGraph的主要优势。"
    }
    
    result = app.invoke(inputs)
    print(f"回答: {result['response']}")

回答: LangGraph 是一个基于 **LangChain** 和 **Streamlit** 构建的框架，专为构建复杂、状态驱动的生成式 AI 应用设计。它的核心目标是解决 AI 应用在状态管理、流程控制和可视化监控方面的挑战。以下是其主要优势的详细分析：

---

### **1. 强大的状态管理能力**
- **长时程状态支持**：LangGraph 允许开发者定义和维护应用的长期状态（如用户会话历史、中间结果等），这对于多轮对话、复杂任务流程（如订单处理、客服流程）至关重要。
- **灵活的状态存储**：支持将状态存储在内存、数据库（如 PostgreSQL）或分布式系统（如 Redis）中，适应不同规模和实时性需求。
- **状态可视化与调试**：通过集成的可视化工具，开发者可以实时监控状态变化，快速定位问题（例如某步骤的上下文丢失或状态异常）。

---

### **2. 模块化与可扩展性**
- **基于节点的工作流**：LangGraph 的核心是 **状态图（StateGraph）**，开发者可以将任务拆分为多个节点（Node），每个节点对应一个具体功能（如意图识别、数据查询、生成响应）。节点之间的连接（Edges）定义了流程逻辑，类似于工作流引擎（如 Apache Airflow）。
- **动态流程控制**：支持根据状态动态调整流程分支（例如，用户提问类型不同，触发不同的处理逻辑），而非硬编码固定流程。
- **与 LangChain 深度集成**：直接兼容 LangChain 的 Chains、Agents、Prompt Templates 等模块，开发者可以复用已有的 AI 组件（如用 LLM 生成答案，用 Vector Store 查询知识库）。

---

### **3. 可视化与监控**
- **实时状态图监控**：通过 Streamlit 的 UI，开发者可以实时查看应用的运行状态（当前节点、状态数据、流程路径），显著降低调试复杂度。
- **交互式调试**：用户可以直接在界面上模拟输入、修改状态，观察流程执行结果，快速验证逻辑。
- **日志与追踪**：自动记录流程中的每一步操作和状态变更，便于后期分析和优化。

---

### **4. 低代码/低耦合开发**
- **声明式 API**：通过定义状态结构、节点函数和